# LeetCode 3906 — Count Good Integers on a Grid Path bằng Python

- Link: https://leetcode.com/problems/count-good-integers-on-a-grid-path/
- Độ khó: **Hard**
- Chủ đề: Digit DP (quy hoạch động theo chữ số), đếm tổ hợp (chia kẹo Euler / tổ hợp lặp), kỹ thuật `f(r) - f(l-1)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countGoodIntegersOnPath(self, l: int, r: int, directions: str) -> int:
```

## 1. Mục tiêu học

1. Đưa bài "đường đi trên lưới 4×4" về điều kiện trên **các vị trí chữ số** của một xâu 16 ký tự (nhận ra đường đi luôn đi theo **chỉ số tăng dần**).
2. Dùng kỹ thuật **đếm trên `[0, N]`** rồi lấy `f(r) - f(l - 1)`.
3. Viết **Digit DP** trạng thái `(vị trí, chữ số chốt cuối cùng, cờ chặn trên)` bằng `functools.cache`.
4. Đếm **không cần DP** bằng công thức tổ hợp lặp: số dãy không giảm độ dài `k` từ `{a, …, 9}` là `C(10 - a + k - 1, k)`.
5. Viết DP dạng bảng (bottom-up) và kiểm thử đối chiếu với duyệt trực tiếp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `l`, `r` và xâu `directions` gồm đúng 3 chữ `'D'` và 3 chữ `'R'`. Với mỗi số nguyên `x ∈ [l, r]`:

1. Đệm số 0 bên trái cho đủ **16 chữ số**.
2. Xếp 16 chữ số vào lưới `4 × 4` theo hàng (4 chữ số đầu là hàng 0, …).
3. Bắt đầu ở ô `(0, 0)`, lần lượt đi theo `directions`: `'D'` xuống một hàng, `'R'` sang phải một cột. Ghi lại **7** chữ số đã đi qua (kể cả ô đầu).
4. `x` **tốt** nếu dãy 7 chữ số đó **không giảm**.

Đếm số `x` tốt trong `[l, r]`.

**Ví dụ 1:** `l = 8, r = 10, directions = "DDDRRR"` → `2` (8 và 9 tốt; 10 cho dãy `0,0,0,0,0,1,0` — giảm ở cuối).

**Ví dụ 2:** `l = 123456789, r = 123456790, directions = "DDRRDR"` → `1`.

**Ví dụ 3:** `l = r = 1288561398769758, directions = "RRRDDD"` → `0`.

**Ràng buộc:** `1 ≤ l ≤ r ≤ 9 × 10^15`, `directions` dài 6, có đúng ba `'D'` và ba `'R'`.

In [1]:
from functools import cache
from math import comb
import random, time

examples = [(8, 10, "DDDRRR", 2), (123456789, 123456790, "DDRRDR", 1),
            (1288561398769758, 1288561398769758, "RRRDDD", 0)]
for l, r, d, exp in examples:
    print(l, r, d, "-> kỳ vọng", exp)

8 10 DDDRRR -> kỳ vọng 2
123456789 123456790 DDRRDR -> kỳ vọng 1
1288561398769758 1288561398769758 RRRDDD -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `str.zfill` và đổi tọa độ lưới ↔ chỉ số

`str(x).zfill(16)` đệm số 0 bên trái. Ô `(row, col)` trong lưới 4×4 xếp theo hàng ứng với chỉ số `row * 4 + col` trong xâu.

In [2]:
s = str(123456789).zfill(16)
print(s)
for row in range(4):
    print(" ".join(s[row * 4 + c] for c in range(4)))
print("ô (2, 3) ->", s[2 * 4 + 3])

0000000123456789
0 0 0 0
0 0 0 1
2 3 4 5
6 7 8 9
ô (2, 3) -> 5


### 3.2. Tính các ô trên đường đi

Mỗi bước `'D'` tăng chỉ số thêm 4, `'R'` tăng thêm 1 — **luôn tăng**. Vì vậy thứ tự đi qua các ô trùng với thứ tự chỉ số trong xâu. Ta chỉ cần biết tập 7 chỉ số "chốt" (key).

In [3]:
def path_cells(directions):
    row = col = 0
    cells = [0]
    for c in directions:
        if c == 'D':
            row += 1
        else:
            col += 1
        cells.append(row * 4 + col)
    return cells

for d in ["DDDRRR", "DDRRDR", "RRRDDD", "RDRDRD"]:
    print(d, path_cells(d))

DDDRRR [0, 4, 8, 12, 13, 14, 15]
DDRRDR [0, 4, 8, 9, 10, 14, 15]
RRRDDD [0, 1, 2, 3, 7, 11, 15]
RDRDRD [0, 1, 5, 6, 10, 11, 15]


### 3.3. `functools.cache` cho đệ quy có nhớ

`@cache` lưu kết quả theo tham số, tránh tính lại. Lưu ý gọi `f.cache_clear()` nếu hàm phụ thuộc biến bên ngoài thay đổi (ở đây xâu giới hạn `s` đổi giữa `f(r)` và `f(l-1)`).

In [4]:
@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(80), fib.cache_info().hits > 0)

23416728348467685 True


### 3.4. Tổ hợp lặp (chia kẹo Euler)

Số dãy **không giảm** độ dài `k` lấy từ tập `t` giá trị là `C(t + k - 1, k)` (chọn multiset `k` phần tử từ `t` loại). Với chữ số từ `a` tới `9`: `t = 10 - a`.

In [5]:
from itertools import combinations_with_replacement
a, k = 3, 4
print(comb(10 - a + k - 1, k), len(list(combinations_with_replacement(range(a, 10), k))))

210 210


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 với `directions = "DDDRRR"`: các ô `0, 4, 8, 12, 13, 14, 15`. Với `x = 10`, xâu `0000000000000010`:

- vị trí 0, 4, 8, 12, 13 đều là `0`;
- vị trí 14 là `1`, vị trí 15 là `0` ⇒ dãy `0,0,0,0,0,1,0` **giảm** ở cuối ⇒ không tốt.

**Quan sát quan trọng:** điều kiện chỉ đặt lên **7 vị trí chốt**, và đòi hỏi chữ số ở các vị trí đó (theo thứ tự trái → phải) **không giảm**. 9 vị trí còn lại tự do.

Nếu không có giới hạn `[l, r]` (đếm mọi xâu 16 chữ số) thì đáp án là `C(10 + 7 - 1, 7) · 10^9 = 11440 · 10^9`. Khó khăn duy nhất là **giới hạn trên** ⇒ Digit DP hoặc đếm theo tiền tố.

In [6]:
print("số xâu 16 chữ số tốt (không giới hạn):", comb(16, 7) * 10**9)

số xâu 16 chữ số tốt (không giới hạn): 11440000000000


## 5. Cách 1 — Brute Force: duyệt từng số

Duyệt `x` từ `l` tới `r`, dựng xâu 16 chữ số, lấy 7 chữ số chốt và kiểm tra không giảm. `O((r - l + 1) · 16)` — chỉ chạy được khi đoạn ngắn.

In [7]:
def is_good(x, cells):
    s = str(x).zfill(16)
    seq = [s[i] for i in cells]
    return all(seq[i] <= seq[i + 1] for i in range(len(seq) - 1))

def brute(l, r, directions):
    cells = path_cells(directions)
    return sum(1 for x in range(l, r + 1) if is_good(x, cells))

for l, r, d, exp in examples:
    print(brute(l, r, d), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Nút thắt:** `r - l` có thể tới `9·10^{15}`. Phải đếm **theo cấu trúc chữ số** thay vì từng số.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Tiền tố của đoạn:** `count(l, r) = f(r) - f(l - 1)` với `f(N)` = số `x ∈ [0, N]` tốt.

**Bước 2 — Digit DP:** đi qua 16 chữ số của `N` (đã đệm), trạng thái:

- `pos`: vị trí đang chọn;
- `last`: chữ số ở vị trí chốt gần nhất đã chọn (ban đầu `0`, vì chữ số ≥ 0);
- `tight`: tiền tố đang **bằng** tiền tố của `N` (nếu đúng, chữ số tiếp theo không được vượt `N[pos]`).

Tại `pos`: chữ số `d` chạy từ `lo` tới `hi`, với `lo = last` nếu `pos` là vị trí chốt, ngược lại `lo = 0`; `hi = N[pos]` nếu `tight`, ngược lại `9`.

**Bước 3 — Đột phá tổ hợp:** khi `tight = False`, phần còn lại là **tự do**: các vị trí không chốt mỗi chỗ 10 lựa chọn, các vị trí chốt (giả sử còn `k` chỗ) là dãy không giảm từ `{last..9}` ⇒ `C(10 - last + k - 1, k)`. Nên ta chỉ cần đi dọc **đường chặn** (tight) và cộng công thức mỗi khi rẽ xuống chữ số nhỏ hơn — `O(16 · 10)`.

## 7. Các cách tiếp theo

- **Cách 2:** Digit DP đệ quy có nhớ `(pos, last, tight)`.
- **Cách 3:** đếm theo tiền tố + **công thức tổ hợp** (không cần memo).
- **Cách 4:** Digit DP **dạng bảng** (bottom-up) với mảng `cnt[last]` cho trạng thái "tự do" và một trạng thái "chặn".

### 7.1. Cách 2 — Digit DP đệ quy

In [8]:
def digit_dp(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions):
        key[c] = True

    def f(N):
        if N < 0:
            return 0
        s = str(N).zfill(16)

        @cache
        def dfs(pos, last, tight):
            if pos == 16:
                return 1
            lo = last if key[pos] else 0
            hi = int(s[pos]) if tight else 9
            total = 0
            for d in range(lo, hi + 1):
                total += dfs(pos + 1, d if key[pos] else last, tight and d == hi)
            return total

        return dfs(0, 0, True)

    return f(r) - f(l - 1)

for l, r, d, exp in examples:
    print(digit_dp(l, r, d), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp Cách 2:** số trạng thái `16 · 10 · 2`, mỗi trạng thái thử ≤ 10 chữ số ⇒ `O(16 · 10 · 10 · 2)` ≈ 3200 phép tính cho mỗi `f`.

### 7.2. Cách 3 — Đếm theo tiền tố + tổ hợp

**Giải thích từng đoạn:**

1. `suffix_keys[pos]` = số vị trí chốt trong `[pos, 16)`; `suffix_free[pos]` = số vị trí tự do.
2. `free_count(pos, last)` = `10^{free} · C(10 - last + k - 1, k)` — số cách điền phần đuôi từ `pos` khi không còn bị chặn.
3. Đi dọc `N`: tại `pos`, với mỗi chữ số `d < N[pos]` hợp lệ (`d ≥ last` nếu chốt), cộng `free_count(pos + 1, last')`. Rồi đặt chữ số `N[pos]`; nếu nó vi phạm (`N[pos] < last` tại vị trí chốt) thì **dừng** (đường chặn chết). Nếu đi hết 16 vị trí thì cộng 1 (chính `N`).

Lưu ý `C(10 - last + k - 1, k)` với `k = 0` bằng `1` (không còn vị trí chốt).

In [9]:
def combinatorial(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions):
        key[c] = True
    suf_k = [0] * 17
    suf_f = [0] * 17
    for p in range(15, -1, -1):
        suf_k[p] = suf_k[p + 1] + key[p]
        suf_f[p] = suf_f[p + 1] + (not key[p])

    def free_count(pos, last):
        k = suf_k[pos]
        return 10 ** suf_f[pos] * comb(10 - last + k - 1, k)

    def f(N):
        if N < 0:
            return 0
        s = str(N).zfill(16)
        total, last = 0, 0
        for pos in range(16):
            cur = int(s[pos])
            lo = last if key[pos] else 0
            for d in range(lo, cur):
                total += free_count(pos + 1, d if key[pos] else last)
            if key[pos]:
                if cur < last:
                    return total          # tiền tố của N đã vi phạm
                last = cur
        return total + 1                  # chính N là số tốt

    return f(r) - f(l - 1)

for l, r, d, exp in examples:
    print(combinatorial(l, r, d), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


In [10]:
class SolutionComb:
    def countGoodIntegersOnPath(self, l: int, r: int, directions: str) -> int:
        return combinatorial(l, r, directions)

print([SolutionComb().countGoodIntegersOnPath(l, r, d) for l, r, d, _ in examples])

[2, 1, 0]


**Độ phức tạp Cách 3:** `O(16 · 10)` phép tính `comb` cho mỗi `f`, bộ nhớ `O(16)`.

### 7.3. Cách 4 — Digit DP dạng bảng (bottom-up)

**Ý tưởng:** duyệt `pos` từ trái sang phải, giữ:

- `free[last]` = số tiền tố **đã nhỏ hơn** `N` (không còn bị chặn) có chữ số chốt cuối là `last`;
- `tight_last` = chữ số chốt cuối của tiền tố **bằng** `N` (hoặc `None` nếu tiền tố đó đã vi phạm).

Chuyển: từ `free`, mọi chữ số hợp lệ; từ trạng thái chặn, chữ số `< N[pos]` đi vào `free`, chữ số `= N[pos]` giữ trạng thái chặn.

In [11]:
def tabulated(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions):
        key[c] = True

    def f(N):
        if N < 0:
            return 0
        s = str(N).zfill(16)
        free = [0] * 10
        tight_last = 0                     # tiền tố rỗng: last = 0
        for pos in range(16):
            cur = int(s[pos])
            nf = [0] * 10
            if key[pos]:
                # từ free: chọn d >= last
                run = 0
                for d in range(10):
                    run += free[d]          # Σ_{last <= d} free[last]
                    nf[d] += run
            else:
                for last in range(10):
                    nf[last] += free[last] * 10
            if tight_last is not None:
                lo = tight_last if key[pos] else 0
                for d in range(lo, cur):
                    nf[d if key[pos] else tight_last] += 1
                if key[pos] and cur < tight_last:
                    tight_last = None
                elif key[pos]:
                    tight_last = cur
            free = nf
        return sum(free) + (1 if tight_last is not None else 0)

    return f(r) - f(l - 1)

for l, r, d, exp in examples:
    print(tabulated(l, r, d), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp Cách 4:** `O(16 · 10)` cho mỗi `f`, bộ nhớ `O(10)`.

## 8. TRACE từng bước

Trace Cách 3 với `N = 10`, `directions = "DDDRRR"` (chốt tại `0, 4, 8, 12, 13, 14, 15`). Xâu `0000000000000010`.

- Vị trí 0..13: chữ số `0`, không có `d < 0` để rẽ; vị trí chốt giữ `last = 0`.
- Vị trí 14 (chốt), `cur = 1`: rẽ `d = 0` ⇒ cộng `free_count(15, 0)` = `C(10 + 1 - 1, 1) = 10` (vị trí 15 chốt, chữ số `≥ 0`). Đặt `last = 1`.
- Vị trí 15 (chốt), `cur = 0 < last = 1` ⇒ đường chặn chết, trả `10`.

Vậy `f(10) = 10` (các số `0..9` đều tốt, `10` không tốt). Tương tự `f(7) = 8` ⇒ đáp án `[8, 10]` là `2` ✔.

In [12]:
def trace_comb(N, directions):
    key = [False] * 16
    for c in path_cells(directions):
        key[c] = True
    suf_k = [0] * 17; suf_f = [0] * 17
    for p in range(15, -1, -1):
        suf_k[p] = suf_k[p + 1] + key[p]; suf_f[p] = suf_f[p + 1] + (not key[p])
    s = str(N).zfill(16); total, last = 0, 0
    print("N =", s, "| chốt:", [p for p in range(16) if key[p]])
    for pos in range(16):
        cur = int(s[pos]); lo = last if key[pos] else 0
        add = 0
        for d in range(lo, cur):
            nl = d if key[pos] else last
            add += 10 ** suf_f[pos + 1] * comb(10 - nl + suf_k[pos + 1] - 1, suf_k[pos + 1])
        total += add
        if add or key[pos] and cur < last:
            print(f"pos={pos:2} {'chốt' if key[pos] else 'tự do'} cur={cur} last={last} -> cộng {add}, tổng={total}")
        if key[pos]:
            if cur < last:
                print("   vi phạm trên đường chặn -> dừng"); return total
            last = cur
    print("đi hết đường chặn -> cộng 1 cho chính N"); return total + 1

print("f(10) =", trace_comb(10, "DDDRRR"))
print("f(7)  =", trace_comb(7, "DDDRRR"))

N = 0000000000000010 | chốt: [0, 4, 8, 12, 13, 14, 15]
pos=14 chốt cur=1 last=0 -> cộng 10, tổng=10
pos=15 chốt cur=0 last=1 -> cộng 0, tổng=10
   vi phạm trên đường chặn -> dừng
f(10) = 10
N = 0000000000000007 | chốt: [0, 4, 8, 12, 13, 14, 15]
pos=15 chốt cur=7 last=0 -> cộng 7, tổng=7
đi hết đường chặn -> cộng 1 cho chính N
f(7)  = 8


In [13]:
# Trace một số lớn hơn để thấy nhiều lần "rẽ": N = 123456789, "DDRRDR"
print("f(123456789) =", trace_comb(123456789, "DDRRDR"))
print("đối chiếu digit DP:", digit_dp(0, 123456789, "DDRRDR"))

N = 0000000123456789 | chốt: [0, 4, 8, 9, 10, 14, 15]
pos= 7 tự do cur=1 last=0 -> cộng 2002000, tổng=2002000
pos= 8 chốt cur=2 last=0 -> cộng 1210000, tổng=3212000
pos= 9 chốt cur=3 last=2 -> cộng 120000, tổng=3332000
pos=10 chốt cur=4 last=3 -> cộng 28000, tổng=3360000
pos=11 tự do cur=5 last=4 -> cộng 10500, tổng=3370500
pos=12 tự do cur=6 last=4 -> cộng 1260, tổng=3371760
pos=13 tự do cur=7 last=4 -> cộng 147, tổng=3371907
pos=14 chốt cur=8 last=4 -> cộng 18, tổng=3371925
pos=15 chốt cur=9 last=8 -> cộng 1, tổng=3371926
đi hết đường chặn -> cộng 1 cho chính N
f(123456789) = 3371927
đối chiếu digit DP: 3371927


## 9. Code nộp LeetCode

In [14]:
from math import comb

class Solution:
    def countGoodIntegersOnPath(self, l: int, r: int, directions: str) -> int:
        key = [False] * 16
        row = col = 0
        key[0] = True
        for c in directions:
            if c == 'D':
                row += 1
            else:
                col += 1
            key[row * 4 + col] = True
        suf_k = [0] * 17
        suf_f = [0] * 17
        for p in range(15, -1, -1):
            suf_k[p] = suf_k[p + 1] + key[p]
            suf_f[p] = suf_f[p + 1] + (not key[p])

        def f(N):
            if N < 0:
                return 0
            s = str(N).zfill(16)
            total, last = 0, 0
            for pos in range(16):
                cur = ord(s[pos]) - 48
                k, fr = suf_k[pos + 1], suf_f[pos + 1]
                if key[pos]:
                    for d in range(last, cur):
                        total += 10 ** fr * comb(10 - d + k - 1, k)
                    if cur < last:
                        return total
                    last = cur
                else:
                    total += cur * 10 ** fr * comb(10 - last + k - 1, k)
            return total + 1

        return f(r) - f(l - 1)

sol = Solution()
for l, r, d, exp in examples:
    got = sol.countGoodIntegersOnPath(l, r, d)
    print(got, "✔" if got == exp else "✘")

2 ✔
1 ✔
0 ✔


## 10. Vì sao thuật toán đúng

1. **Đường đi = thứ tự chỉ số:** mỗi bước `'D'` cộng 4 và `'R'` cộng 1 vào chỉ số ô, nên dãy chỉ số trên đường đi tăng ngặt. "Dãy chữ số trên đường đi không giảm" ⇔ "các chữ số ở 7 vị trí chốt, đọc trái → phải, không giảm".
2. **`f(r) - f(l-1)`:** đếm `[0, r]` trừ `[0, l-1]` cho đúng `[l, r]`; số `0` (xâu toàn 0) được tính trong cả hai nên triệt tiêu.
3. **Phân hoạch theo vị trí rẽ đầu tiên:** mọi `x < N` (16 chữ số) có duy nhất vị trí đầu tiên `pos` mà `x[pos] < N[pos]`, với tiền tố trước đó bằng `N`. Cách 3 cộng đúng mỗi nhóm một lần; trong nhóm đó phần đuôi tự do, đếm bằng tích độc lập: vị trí không chốt 10 cách, vị trí chốt là dãy không giảm `≥ last` (tổ hợp lặp). Nếu tiền tố của `N` đã vi phạm thì mọi `x` có tiền tố đó đều xấu ⇒ dừng.
4. **Digit DP** (Cách 2, 4) liệt kê cùng phân hoạch đó nhưng cộng dần bằng DP thay vì công thức đóng.

## 11. Phân tích Time / Space Complexity

Gọi `D = 16` (số chữ số), `B = 10` (cơ số).

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng số | `O((r - l) · D)` | `O(D)` |
| 2. Digit DP đệ quy | `O(D · B · B · 2)` | `O(D · B · 2)` |
| 3. Tiền tố + tổ hợp | `O(D · B)` | `O(D)` |
| 4. Digit DP dạng bảng | `O(D · B)` | `O(B)` |

Mọi cách 2–4 đều là hằng số nhỏ (vài nghìn phép tính).

## 12. Bộ test

In [15]:
def run_all(l, r, d, with_brute=True):
    res = [digit_dp(l, r, d), combinatorial(l, r, d), tabulated(l, r, d), Solution().countGoodIntegersOnPath(l, r, d)]
    if with_brute:
        res.append(brute(l, r, d))
    return res

for l, r, d, exp in examples:
    assert all(v == exp for v in run_all(l, r, d)), run_all(l, r, d)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [16]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng duyệt hoặc công thức)
assert Solution().countGoodIntegersOnPath(1, 1, "DDDRRR") == 1                  # số 1: dãy 0..0,1
assert Solution().countGoodIntegersOnPath(1, 9 * 10**15, "RRRDDD") == run_all(1, 9 * 10**15, "RRRDDD", False)[0]
# cả 10^16 xâu (0..10^16-1) có 11440*10^9 xâu tốt; trừ số 0
full = Solution().countGoodIntegersOnPath(1, 10**16 - 1, "DRDRDR")
assert full == comb(16, 7) * 10**9 - 1
print("Đếm toàn bộ 1..10^16-1:", full)
# chữ số cuối là chốt và cực đại
assert Solution().countGoodIntegersOnPath(9 * 10**15, 9 * 10**15, "RRRDDD") == brute(9 * 10**15, 9 * 10**15, "RRRDDD")
print("Ca biên: OK")

Đếm toàn bộ 1..10^16-1: 11439999999999
Ca biên: OK


In [17]:
# 12.3 phản ví dụ: hiểu nhầm "đường đi" là mọi ô trong hàng/cột đi qua (dùng 16 vị trí)
def wrong_all_digits(l, r):
    return sum(1 for x in range(l, r + 1) if list(str(x).zfill(16)) == sorted(str(x).zfill(16)))
print("hiểu sai (toàn bộ chữ số không giảm):", wrong_all_digits(1, 300), "| đúng (RRRDDD):", Solution().countGoodIntegersOnPath(1, 300, "RRRDDD"))

hiểu sai (toàn bộ chữ số không giảm): 135 | đúng (RRRDDD): 300


In [18]:
# 12.4 đối chiếu ngẫu nhiên với duyệt trực tiếp
rng = random.Random(3906)
dirs_all = sorted({"".join(p) for p in __import__('itertools').permutations("DDDRRR")})
for _ in range(150):
    d = rng.choice(dirs_all)
    l = rng.randint(1, 10**rng.randint(1, 15))
    r = l + rng.randint(0, 2000)
    vals = run_all(l, r, d)
    assert len(set(vals)) == 1, (l, r, d, vals)
print("150 test ngẫu nhiên (đoạn ngắn): OK; số hướng đi khác nhau:", len(dirs_all))

150 test ngẫu nhiên (đoạn ngắn): OK; số hướng đi khác nhau: 20


In [19]:
# 12.5 đối chiếu các cách nhanh trên đoạn lớn tùy ý (không duyệt được)
rng = random.Random(7)
for _ in range(300):
    d = rng.choice(dirs_all)
    l = rng.randint(1, 9 * 10**15); r = rng.randint(l, 9 * 10**15)
    vals = run_all(l, r, d, with_brute=False)
    assert len(set(vals)) == 1
t0 = time.time()
for _ in range(1000):
    Solution().countGoodIntegersOnPath(1, 9 * 10**15, "RDRDRD")
print("300 đoạn lớn khớp nhau; 1000 lần gọi mất", round(time.time() - t0, 3), "s")

300 đoạn lớn khớp nhau; 1000 lần gọi mất 0.006 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đệm đủ 16 chữ số

Nếu dùng `str(x)` không đệm, vị trí của chữ số thay đổi theo độ dài số ⇒ đọc sai ô.

In [20]:
def wrong_no_pad(x, cells):
    s = str(x)                                     # SAI: không zfill(16)
    seq = [s[i] for i in cells if i < len(s)]
    return all(seq[i] <= seq[i + 1] for i in range(len(seq) - 1))

cells = path_cells("DDDRRR")
print("x=10: sai ->", wrong_no_pad(10, cells), "| đúng ->", is_good(10, cells))

x=10: sai -> True | đúng -> False


### Lỗi 2 — Áp ràng buộc `last` cho cả vị trí không chốt

In [21]:
def wrong_all_constrained(l, r, directions):
    def f(N):
        if N < 0: return 0
        s = str(N).zfill(16)
        @cache
        def dfs(pos, last, tight):
            if pos == 16: return 1
            hi = int(s[pos]) if tight else 9
            return sum(dfs(pos + 1, d, tight and d == hi) for d in range(last, hi + 1))   # SAI
        return dfs(0, 0, True)
    return f(r) - f(l - 1)

print("sai:", wrong_all_constrained(1, 1000, "RRRDDD"), "| đúng:", Solution().countGoodIntegersOnPath(1, 1000, "RRRDDD"))

sai: 219 | đúng: 1000


**Vì sao sai:** chỉ 7 ô trên đường đi phải không giảm; 9 ô còn lại tự do. **Sửa:** `lo = last if key[pos] else 0` và chỉ cập nhật `last` ở vị trí chốt.

### Lỗi 3 — Không `cache_clear()` (hoặc dùng chung cache) giữa `f(r)` và `f(l-1)`

Nếu hàm đệ quy có nhớ đọc xâu giới hạn `s` từ biến ngoài, cache cũ của `f(r)` sẽ trả kết quả sai cho `f(l-1)`.

In [22]:
def wrong_shared_cache(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    state = {"s": ""}
    @cache
    def dfs(pos, last, tight):
        if pos == 16: return 1
        s = state["s"]
        lo = last if key[pos] else 0
        hi = int(s[pos]) if tight else 9
        return sum(dfs(pos + 1, d if key[pos] else last, tight and d == hi) for d in range(lo, hi + 1))
    def f(N):
        if N < 0: return 0
        state["s"] = str(N).zfill(16)              # SAI: không xóa cache
        return dfs(0, 0, True)
    return f(r) - f(l - 1)

print("sai:", wrong_shared_cache(8, 10, "DDDRRR"), "| đúng:", Solution().countGoodIntegersOnPath(8, 10, "DDDRRR"))

sai: 0 | đúng: 2


### Lỗi 4 — Quên cộng chính `N` (hoặc cộng dù `N` không tốt)

Ở Cách 3, sau khi đi hết đường chặn mà không vi phạm thì `N` tốt ⇒ `+1`. Nếu luôn `+1` (kể cả khi vi phạm) thì đếm dư.

In [23]:
def wrong_always_plus_one(N, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    s = str(N).zfill(16); total, last = 0, 0
    suf_k = [sum(key[p:]) for p in range(17)]; suf_f = [16 - p - suf_k[p] for p in range(17)]
    for pos in range(16):
        cur = int(s[pos]); lo = last if key[pos] else 0
        for d in range(lo, cur):
            nl = d if key[pos] else last
            total += 10 ** suf_f[pos + 1] * comb(10 - nl + suf_k[pos + 1] - 1, suf_k[pos + 1])
        if key[pos]:
            if cur < last: break                   # thoát vòng...
            last = cur
    return total + 1                               # SAI: luôn cộng 1

print("f(10) sai:", wrong_always_plus_one(10, "DDDRRR"), "| đúng:", digit_dp(0, 10, "DDDRRR"))

f(10) sai: 11 | đúng: 10


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đếm số trong `[l, r]` thỏa tính chất về chữ số**, `r` tới `10^{15}–10^{18}` ⇒ **Digit DP** với `f(r) - f(l - 1)`.
- Trạng thái digit DP = (vị trí, **thông tin tối thiểu** để kiểm tra phần còn lại, cờ `tight`). Ở đây thông tin là chữ số chốt cuối.
- Khi phần "tự do" có **công thức đóng** (tích các lựa chọn độc lập, tổ hợp lặp) ⇒ bỏ memo, chỉ đi dọc đường chặn.
- **Lưới + đường đi chỉ xuống/phải** ⇒ thứ tự thăm = thứ tự chỉ số theo hàng (row-major). Bài "lưới" thực ra là bài "xâu".

## 15. Biến thể của bài

1. **Tăng ngặt** thay vì không giảm: lo = `last + 1` ở vị trí chốt (và trạng thái ban đầu `last = -1`), tổ hợp thường `C(9 - last, k)`.
2. **Đường đi tùy ý (cả lên/trái)**: thứ tự thăm không còn trùng thứ tự chỉ số ⇒ trạng thái DP phải nhớ chữ số tại các ô chốt chưa "đến lượt".
3. **Lưới `a × b` khác**: đổi `16` thành `a·b` và bước `'D'` cộng `b`.
4. **Đếm tổng các số tốt** thay vì số lượng: DP lưu thêm tổng.
5. Liên quan: *Count Numbers with Unique Digits*, *Numbers With Repeated Digits* (LC 1012), *Count of Integers* (LC 2719).

In [24]:
# Biến thể 1: tăng ngặt trên đường đi
def strict_variant(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    def f(N):
        if N < 0: return 0
        s = str(N).zfill(16)
        @cache
        def dfs(pos, last, tight):
            if pos == 16: return 1
            lo = last + 1 if key[pos] else 0
            hi = int(s[pos]) if tight else 9
            return sum(dfs(pos + 1, d if key[pos] else last, tight and d == hi) for d in range(lo, hi + 1))
        return dfs(0, -1, True)
    return f(r) - f(l - 1)

def brute_strict(l, r, directions):
    cells = path_cells(directions); cnt = 0
    for x in range(l, r + 1):
        s = str(x).zfill(16); seq = [s[i] for i in cells]
        cnt += all(seq[i] < seq[i + 1] for i in range(6))
    return cnt

rng = random.Random(15)
for _ in range(40):
    d = rng.choice(dirs_all); l = rng.randint(1, 10**15); r = l + rng.randint(0, 3000)
    assert strict_variant(l, r, d) == brute_strict(l, r, d)
print("Biến thể tăng ngặt khớp duyệt; toàn bộ:", strict_variant(0, 10**16 - 1, "DRDRDR"), "= C(10,7)*10^9 =", comb(10, 7) * 10**9)

Biến thể tăng ngặt khớp duyệt; toàn bộ: 120000000000 = C(10,7)*10^9 = 120000000000


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt | Kiểm tra từng số | `O((r-l)·D)` | `O(D)` | Kiểm thử, đoạn ngắn |
| 2. Digit DP đệ quy | `(pos, last, tight)` + memo | `O(D B² )` | `O(D B)` | Khuôn mẫu chung, dễ mở rộng |
| 3. Tiền tố + tổ hợp | Rẽ tại vị trí đầu tiên, đuôi tự do có công thức | `O(D B)` | `O(D)` | **Lời giải chuẩn**, nhanh nhất |
| 4. DP dạng bảng | `free[last]` + trạng thái chặn | `O(D B)` | `O(B)` | Không muốn đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countGoodIntegersOnPath(1, 20, "RRRDDD")` = ?

**Bài 2 (tính tay).** Đếm số xâu 16 chữ số tốt (không giới hạn) với điều kiện chữ số đầu tiên (ô `(0,0)`) **bằng 5**.

**Bài 3 (điền TODO).** Viết `count_upto(N, directions)` theo Cách 3 (không xem code).

**Bài 4 (tự viết lại).** Viết digit DP đệ quy nhưng trạng thái thêm cờ `started` (bỏ qua số 0 đầu) và chứng minh bằng test rằng ở bài này cờ đó **không cần**.

**Bài 5 (mini challenge).** Đếm số `x ∈ [l, r]` mà dãy trên đường đi **không tăng** (ngược lại). Kiểm chứng với duyệt.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_upto(N, directions):
    # TODO: key, suf_k, suf_f; đi dọc N, cộng 10^free * C(10-last+k-1, k) khi rẽ
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_nonincreasing(l, r, directions):
    # TODO: digit DP với hi ở vị trí chốt bị chặn bởi last (ban đầu last = 9)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chốt `0,1,2,3,7,11,15`. (2) Với `x ≤ 20`, các vị trí `0..13` đều là `0`; chỉ vị trí 15 (chốt) và 14 (tự do) thay đổi. (3) Mọi số `1..20` đều tốt ⇒ `20`.

**Bài 2.** (1) Vị trí 0 cố định bằng 5. (2) 6 chốt còn lại không giảm từ `{5..9}`: `C(5 + 6 - 1, 6) = 210`. (3) Nhân `10^9` ⇒ `210 · 10^9`.

**Bài 3.** (1) Như mục 7.2. (2) `+1` khi đi hết. (3) Đối chiếu `digit_dp(0, N, d)`.

**Bài 4.** (1) Số 0 đầu cũng là chữ số `0` của lưới. (2) Vì `0` là nhỏ nhất, không ảnh hưởng tính không giảm. (3) Test ngẫu nhiên.

**Bài 5.** (1) Ở vị trí chốt chữ số `≤ last`. (2) Ban đầu `last = 9`. (3) Đối chiếu duyệt.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().countGoodIntegersOnPath(1, 20, "RRRDDD") == 20 == brute(1, 20, "RRRDDD")
cnt = Solution().countGoodIntegersOnPath(5 * 10**15, 6 * 10**15 - 1, "DRDRDR")
assert cnt == comb(10, 6) * 10**9 == 210 * 10**9
print("Bài 1: 20 | Bài 2:", cnt)

Bài 1: 20 | Bài 2: 210000000000


In [28]:
# Bài 3
def count_upto_sol(N, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    suf_k = [0] * 17; suf_f = [0] * 17
    for p in range(15, -1, -1):
        suf_k[p] = suf_k[p + 1] + key[p]; suf_f[p] = suf_f[p + 1] + (not key[p])
    s = str(N).zfill(16); total = last = 0
    for pos in range(16):
        cur = int(s[pos]); lo = last if key[pos] else 0
        for d in range(lo, cur):
            nl = d if key[pos] else last
            total += 10 ** suf_f[pos + 1] * comb(10 - nl + suf_k[pos + 1] - 1, suf_k[pos + 1])
        if key[pos]:
            if cur < last: return total
            last = cur
    return total + 1

rng = random.Random(3)
for _ in range(200):
    d = rng.choice(dirs_all); N = rng.randint(0, 9 * 10**15)
    assert count_upto_sol(N, d) == digit_dp(0, N, d)
print("Bài 3 OK")

Bài 3 OK


In [29]:
# Bài 4
def dp_with_started(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    def f(N):
        if N < 0: return 0
        s = str(N).zfill(16)
        @cache
        def dfs(pos, last, tight, started):
            if pos == 16: return 1
            lo = last if key[pos] else 0
            hi = int(s[pos]) if tight else 9
            return sum(dfs(pos + 1, d if key[pos] else last, tight and d == hi, started or d > 0) for d in range(lo, hi + 1))
        return dfs(0, 0, True, False)
    return f(r) - f(l - 1)

rng = random.Random(44)
for _ in range(100):
    d = rng.choice(dirs_all); l = rng.randint(1, 10**15); r = rng.randint(l, 9 * 10**15)
    assert dp_with_started(l, r, d) == Solution().countGoodIntegersOnPath(l, r, d)
print("Bài 4 OK: cờ started không làm thay đổi kết quả")

Bài 4 OK: cờ started không làm thay đổi kết quả


In [30]:
# Bài 5
def count_nonincreasing_sol(l, r, directions):
    key = [False] * 16
    for c in path_cells(directions): key[c] = True
    def f(N):
        if N < 0: return 0
        s = str(N).zfill(16)
        @cache
        def dfs(pos, last, tight):
            if pos == 16: return 1
            hi = int(s[pos]) if tight else 9
            top = min(hi, last) if key[pos] else hi
            return sum(dfs(pos + 1, d if key[pos] else last, tight and d == hi) for d in range(0, top + 1))
        return dfs(0, 9, True)
    return f(r) - f(l - 1)

def brute_noninc(l, r, directions):
    cells = path_cells(directions); c = 0
    for x in range(l, r + 1):
        s = str(x).zfill(16); seq = [s[i] for i in cells]
        c += all(seq[i] >= seq[i + 1] for i in range(6))
    return c

rng = random.Random(5)
for _ in range(40):
    d = rng.choice(dirs_all); l = rng.randint(1, 10**15); r = l + rng.randint(0, 3000)
    assert count_nonincreasing_sol(l, r, d) == brute_noninc(l, r, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Digit DP khuôn mẫu
def f(N):
    if N < 0: return 0
    s = str(N).zfill(D)
    @cache
    def dfs(pos, state, tight):
        if pos == D: return 1 if hợp_lệ(state) else 0
        hi = int(s[pos]) if tight else 9
        return sum(dfs(pos+1, chuyển(state, d), tight and d == hi)
                   for d in range(lo(state, pos), hi+1))
    return dfs(0, init, True)
answer = f(r) - f(l-1)
```

**Checklist:**

- [ ] Đã đệm đủ 16 chữ số (`zfill(16)`)?
- [ ] Đã tính đúng 7 vị trí chốt (`'D'` +4, `'R'` +1, gồm ô đầu `0`)?
- [ ] Ràng buộc `≥ last` **chỉ** ở vị trí chốt?
- [ ] `f(r) - f(l - 1)` với `f(-1) = 0`?
- [ ] Xóa cache giữa hai lần gọi (nếu dùng memo theo biến ngoài)?
- [ ] Chỉ `+1` cho chính `N` khi đường chặn không vi phạm?